# Comprehensive EDA Pipeline
## Automated Data Profiling & Report Generation

**Supported Formats:** CSV, Excel, JSON, Parquet, Feather, Pickle, TSV

This notebook automatically generates:
- YData Profiling HTML Report
- Sweetviz HTML Report
- AutoViz Visualizations
- D-Tale Interactive Dashboard (in-notebook)

---

## 1. Import Libraries & Configuration

In [18]:
import pandas as pd
import numpy as np
import os
import json
import sys
import warnings
import logging
from pathlib import Path
from datetime import datetime
import chardet
import traceback

warnings.filterwarnings('ignore')

# CONFIGURATION - MODIFY THIS
DATA_FILE = 'datasets/wine+quality/winequality-red-edited.csv'  # ← CHANGE THIS TO YOUR FILE
OUTPUT_DIR = 'reports'
CORRELATION_THRESHOLD = 0.5

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✓ Libraries loaded')

✓ Libraries loaded


## 2. Helper Functions

In [19]:
def detect_encoding(file_path):
    try:
        with open(file_path, 'rb') as f:
            result = chardet.detect(f.read(10000))
        return result['encoding'] or 'utf-8'
    except:
        return 'utf-8'

def detect_delimiter(file_path, encoding):
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            sample = f.read(10000)
        delimiters = [',', ';', '\t', '|']
        return max(delimiters, key=sample.count)
    except:
        return ','

def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'File not found: {file_path}')
    
    ext = os.path.splitext(file_path)[1].lower()
    
    try:
        if ext in ['.csv', '.tsv']:
            enc = detect_encoding(file_path)
            delim = detect_delimiter(file_path, enc)
            df = pd.read_csv(file_path, encoding=enc, sep=delim)
        elif ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
        elif ext == '.json':
            df = pd.read_json(file_path)
        elif ext == '.parquet':
            df = pd.read_parquet(file_path)
        elif ext == '.feather':
            df = pd.read_feather(file_path)
        elif ext in ['.pkl', '.pickle']:
            df = pd.read_pickle(file_path)
        else:
            raise ValueError(f'Unsupported format: {ext}')
        return df
    except Exception as e:
        print(f'Error loading data: {e}')
        raise

print('✓ Helper functions ready')

✓ Helper functions ready


## 3. Load Dataset

In [20]:
start_time = datetime.now()

try:
    df = load_data(DATA_FILE)
    dataset_name = os.path.splitext(os.path.basename(DATA_FILE))[0]
    dataset_output_dir = os.path.join(OUTPUT_DIR, dataset_name)
    Path(dataset_output_dir).mkdir(parents=True, exist_ok=True)
    
    print(f'✓ Data loaded successfully!')
    print(f'  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
    print(f'  Output: {dataset_output_dir}')
except Exception as e:
    print(f'✗ Failed to load: {e}')
    sys.exit(1)

✓ Data loaded successfully!
  Shape: 1,359 rows × 12 columns
  Memory: 0.12 MB
  Output: reports/winequality-red-edited


## 4. Dataset Overview

In [21]:
print('\n' + '='*80)
print('DATASET OVERVIEW')
print('='*80)
print(f'\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\nColumn Information:')
print(df.dtypes)

print('\nFirst 5 rows:')
df.head()


DATASET OVERVIEW

Shape: 1,359 rows × 12 columns
Memory Usage: 0.12 MB

Column Information:
fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sulfur_dioxide     float64
total_sulfur_dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

First 5 rows:


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,3
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,3
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,3
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,4
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,3


## 5. Data Types & Missing Values

In [22]:
print('\n' + '='*80)
print('DATA TYPES & MISSING VALUES')
print('='*80)

info_df = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.values,
    'Non-Null': df.count().values,
    'Null': df.isnull().sum().values,
    'Null %': (df.isnull().sum().values / len(df) * 100).round(2)
})

print('\n', info_df.to_string(index=False))
print(f'\nTotal Missing Values: {df.isnull().sum().sum()}')


DATA TYPES & MISSING VALUES

        Column          Type   Non-Null  Null  Null %
       fixed_acidity float64    1359     0     0.0  
    volatile_acidity float64    1359     0     0.0  
         citric_acid float64    1359     0     0.0  
      residual_sugar float64    1359     0     0.0  
           chlorides float64    1359     0     0.0  
 free_sulfur_dioxide float64    1359     0     0.0  
total_sulfur_dioxide float64    1359     0     0.0  
             density float64    1359     0     0.0  
                  pH float64    1359     0     0.0  
           sulphates float64    1359     0     0.0  
             alcohol float64    1359     0     0.0  
             quality   int64    1359     0     0.0  

Total Missing Values: 0


## 6. Numerical Statistics

In [23]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numerical_cols:
    print('\n' + '='*80)
    print('NUMERICAL STATISTICS')
    print('='*80)
    stats = df[numerical_cols].describe(include='all').T
    stats['skew'] = df[numerical_cols].skew()
    stats['kurtosis'] = df[numerical_cols].kurtosis()
    print('\n', stats.round(4))
else:
    print('\nNo numerical columns found.')


NUMERICAL STATISTICS

                        count   mean      std      min     25%      50%    \
fixed_acidity         1359.0   8.3106   1.7370  4.6000   7.1000   7.9000   
volatile_acidity      1359.0   0.5295   0.1830  0.1200   0.3900   0.5200   
citric_acid           1359.0   0.2723   0.1955  0.0000   0.0900   0.2600   
residual_sugar        1359.0   2.5234   1.3523  0.9000   1.9000   2.2000   
chlorides             1359.0   0.0881   0.0494  0.0120   0.0700   0.0790   
free_sulfur_dioxide   1359.0  15.8933  10.4473  1.0000   7.0000  14.0000   
total_sulfur_dioxide  1359.0  46.8260  33.4089  6.0000  22.0000  38.0000   
density               1359.0   0.9967   0.0019  0.9901   0.9956   0.9967   
pH                    1359.0   3.3098   0.1550  2.7400   3.2100   3.3100   
sulphates             1359.0   0.6587   0.1707  0.3300   0.5500   0.6200   
alcohol               1359.0  10.4323   1.0821  8.4000   9.5000  10.2000   
quality               1359.0   3.6233   0.8236  1.0000   3.0000 

## 7. Categorical Statistics

In [24]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_cols:
    print('\n' + '='*80)
    print('CATEGORICAL STATISTICS')
    print('='*80)
    for col in categorical_cols:
        print(f'\n{col}:')
        print(f'  Unique: {df[col].nunique()}')
        print(f'  Top 5 values:')
        print(df[col].value_counts().head().to_string())
else:
    print('\nNo categorical columns found.')


No categorical columns found.


## 8. Correlation Matrix

In [25]:
if len(numerical_cols) > 1:
    print('\n' + '='*80)
    print('CORRELATION MATRIX')
    print('='*80)
    corr_matrix = df[numerical_cols].corr()
    print('\nFull Correlation Matrix:')
    print(corr_matrix.round(3))
    
    print(f'\nHigh Correlations (abs > {CORRELATION_THRESHOLD}):')
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > CORRELATION_THRESHOLD:
                high_corr.append({
                    'Column 1': corr_matrix.columns[i],
                    'Column 2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    if high_corr:
        print(pd.DataFrame(high_corr).to_string(index=False))
    else:
        print(f'No correlations above {CORRELATION_THRESHOLD} threshold.')
else:
    print('\nNot enough numerical columns for correlation analysis.')


CORRELATION MATRIX

Full Correlation Matrix:
                      fixed_acidity  volatile_acidity  citric_acid  \
fixed_acidity             1.000           -0.255          0.667      
volatile_acidity         -0.255            1.000         -0.551      
citric_acid               0.667           -0.551          1.000      
residual_sugar            0.111           -0.002          0.144      
chlorides                 0.086            0.055          0.210      
free_sulfur_dioxide      -0.141           -0.021         -0.048      
total_sulfur_dioxide     -0.104            0.072          0.047      
density                   0.670            0.024          0.358      
pH                       -0.687            0.247         -0.550      
sulphates                 0.190           -0.257          0.326      
alcohol                  -0.062           -0.198          0.105      
quality                   0.119           -0.395          0.228      

                      residual_sugar  chlor

## 9. Generate panda profiling

In [26]:
print('\n' + '='*80)
print('DATA PROFILING REPORT')
print('='*80)

try:
    print('\nGenerating data profile summary...')
    
    profile_file = os.path.join(dataset_output_dir, 'data_profile_report.html')
    
    html_content = f'''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Data Profile Report - {dataset_name}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }}
            .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 20px; border-radius: 8px; }}
            h1 {{ color: #333; border-bottom: 3px solid #007bff; padding-bottom: 10px; }}
            h2 {{ color: #555; margin-top: 30px; }}
            table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
            th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
            th {{ background-color: #007bff; color: white; }}
            tr:hover {{ background-color: #f9f9f9; }}
            .stat-box {{ display: inline-block; margin: 10px; padding: 15px; background: #f9f9f9; border-left: 4px solid #007bff; }}
            .section {{ margin: 20px 0; padding: 15px; background: #f9f9f9; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>📊 Data Profile Report: {dataset_name}</h1>
            
            <div class="section">
                <h2>📈 Dataset Overview</h2>
                <div class="stat-box">
                    <strong>Rows:</strong> {df.shape[0]:,}
                </div>
                <div class="stat-box">
                    <strong>Columns:</strong> {df.shape[1]}
                </div>
                <div class="stat-box">
                    <strong>Memory:</strong> {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB
                </div>
                <div class="stat-box">
                    <strong>Missing Values:</strong> {df.isnull().sum().sum()}
                </div>
            </div>
            
            <h2>📋 Column Information</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Type</th>
                    <th>Non-Null</th>
                    <th>Null</th>
                    <th>Null %</th>
                </tr>
    '''
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].dtype}</td>
                    <td>{df[col].count():,}</td>
                    <td>{null_count}</td>
                    <td>{null_pct:.2f}%</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <h2>🔢 Numerical Statistics</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Mean</th>
                    <th>Std</th>
                    <th>Min</th>
                    <th>Max</th>
                </tr>
    '''
    
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numerical_cols:
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].mean():.4f}</td>
                    <td>{df[col].std():.4f}</td>
                    <td>{df[col].min():.4f}</td>
                    <td>{df[col].max():.4f}</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <h2>📊 Top Values (Categorical)</h2>
            <table>
                <tr>
                    <th>Column</th>
                    <th>Unique</th>
                    <th>Top Value</th>
                    <th>Frequency</th>
                </tr>
    '''
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in categorical_cols:
        top_val = df[col].value_counts().index[0] if len(df[col].value_counts()) > 0 else 'N/A'
        top_count = df[col].value_counts().values[0] if len(df[col].value_counts()) > 0 else 0
        html_content += f'''
                <tr>
                    <td>{col}</td>
                    <td>{df[col].nunique()}</td>
                    <td>{top_val}</td>
                    <td>{top_count}</td>
                </tr>
        '''
    
    html_content += '''
            </table>
            
            <footer style="margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; text-align: center; color: #666;">
                <p>Generated automatically using Python Pandas</p>
            </footer>
        </div>
    </body>
    </html>
    '''
    
    with open(profile_file, 'w') as f:
        f.write(html_content)
    
    print(f'✓ Custom profiling report saved: {profile_file}')
    if os.path.exists(profile_file):
        file_size_mb = os.path.getsize(profile_file) / 1024**2
        print(f'  File size: {file_size_mb:.2f} MB')
        print(f'  Open in browser: {profile_file}')

except Exception as e:
    print(f'⚠ Error: {str(e)[:200]}')
    import traceback
    print(traceback.format_exc()[:300])


DATA PROFILING REPORT

Generating data profile summary...
✓ Custom profiling report saved: reports/winequality-red-edited/data_profile_report.html
  File size: 0.01 MB
  Open in browser: reports/winequality-red-edited/data_profile_report.html


## 10. Generate Sweetviz Report

In [27]:
import os
import sweetviz as sv
import warnings

warnings.filterwarnings("ignore")

print("\n" + "=" * 80)
print("SWEETVIZ REPORT")
print("=" * 80)

try:
    print("\nGenerating Sweetviz report...")

    # Create report
    my_report = sv.analyze(df)

    # Output path
    sweetviz_file = os.path.join(dataset_output_dir, "sweetviz_report.html")

    # Save HTML report (correct method)
    my_report.show_html(
        filepath=sweetviz_file,
        open_browser=False,
        layout="widescreen",
        scale=1.0
    )

    print(f"✓ Sweetviz report saved: {sweetviz_file}")

    # File validation
    if os.path.exists(sweetviz_file):
        size_mb = os.path.getsize(sweetviz_file) / (1024 ** 2)
        print(f"  File size: {size_mb:.2f} MB")
    else:
        print("⚠ File was not created properly")

except ImportError:
    print("⚠ Sweetviz not installed.")
    print("Install using: pip install sweetviz")

except Exception as e:
    print(f"⚠ Error generating Sweetviz report:\n{e}")

    # Safe fallback (FIXED)
    try:
        print("\nTrying fallback method...")

        sweetviz_file = os.path.join(dataset_output_dir, "sweetviz_report_fallback.html")

        my_report.show_html(
            filepath=sweetviz_file,
            open_browser=False
        )

        print(f"✓ Fallback report saved: {sweetviz_file}")

    except Exception as e2:
        print(f"❌ Fallback also failed: {e2}")


SWEETVIZ REPORT

Generating Sweetviz report...


                                             |          | [  0%]   00:00 -> (? left)

Report reports/winequality-red-edited/sweetviz_report.html was generated.
✓ Sweetviz report saved: reports/winequality-red-edited/sweetviz_report.html
  File size: 1.29 MB


## 11. Generate AutoViz Visualizations

In [34]:
import os
from pathlib import Path
import warnings
from autoviz.AutoViz_Class import AutoViz_Class

warnings.filterwarnings("ignore")

print("\n" + "=" * 80)
print("AUTOVIZ VISUALIZATIONS")
print("=" * 80)

try:
    print("\nGenerating AutoViz plots...")

    autoviz_dir = os.path.join(dataset_output_dir, "autoviz_plots")
    Path(autoviz_dir).mkdir(parents=True, exist_ok=True)

    AV = AutoViz_Class()

    dft = AV.AutoViz(
        filename=DATA_FILE,
        sep=",",
        depVar="",
        dfte=None,
        header=0,
        verbose=1,
        lowess=False,
        chart_format="png",
        max_rows_analyzed=len(df),
        max_cols_analyzed=len(df.columns),
        save_plot_dir=autoviz_dir
    )

    print("\n✓ AutoViz execution completed")

except ImportError:
    print("⚠ AutoViz is not installed.")

except Exception as e:
    print(f"⚠ Error during AutoViz execution:\n{e}")


AUTOVIZ VISUALIZATIONS

Generating AutoViz plots...
Shape of your Data Set loaded: (1359, 12)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  11
    Number of Integer-Categorical Columns =  1
    Number of String-Categorical Columns =  0
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  0
    Number of Numeric-Boolean Columns =  0
    Number of Discrete String Columns =  0
    Number of NLP String Columns =  0
    Number of Date Time Columns =  0
    Number of ID Columns =  0
    Number of Columns to Delete =  0
    12 Predictors classified...
        No variables removed since no ID or low-information variables found in data set
Since Number of Rows in data 1359 excee

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
fixed_acidity,float64,0.000000,NA,4.600000,15.900000,Column has 41 outliers greater than upper bound (12.35) or lower than lower bound(3.95). Cap them or remove them.
volatile_acidity,float64,0.000000,NA,0.120000,1.580000,Column has 19 outliers greater than upper bound (1.02) or lower than lower bound(0.02). Cap them or remove them.
citric_acid,float64,0.000000,NA,0.000000,1.000000,Column has 1 outliers greater than upper bound (0.94) or lower than lower bound(-0.42). Cap them or remove them.
residual_sugar,float64,0.000000,NA,0.900000,15.500000,Column has 126 outliers greater than upper bound (3.65) or lower than lower bound(0.85). Cap them or remove them.
chlorides,float64,0.000000,NA,0.012000,0.611000,Column has 87 outliers greater than upper bound (0.12) or lower than lower bound(0.04). Cap them or remove them.
free_sulfur_dioxide,float64,0.000000,NA,1.000000,72.000000,Column has 26 outliers greater than upper bound (42.00) or lower than lower bound(-14.00). Cap them or remove them.
total_sulfur_dioxide,float64,0.000000,NA,6.000000,289.000000,Column has 45 outliers greater than upper bound (124.50) or lower than lower bound(-39.50). Cap them or remove them.
density,float64,0.000000,NA,0.990070,1.003690,Column has 35 outliers greater than upper bound (1.00) or lower than lower bound(0.99). Cap them or remove them.
pH,float64,0.000000,NA,2.740000,4.010000,Column has 28 outliers greater than upper bound (3.68) or lower than lower bound(2.92). Cap them or remove them.
sulphates,float64,0.000000,NA,0.330000,2.000000,Column has 55 outliers greater than upper bound (1.00) or lower than lower bound(0.28). Cap them or remove them.


Number of All Scatter Plots = 66
All Plots done
Time to run AutoViz = 11 seconds 

 ###################### AUTO VISUALIZATION Completed ########################

✓ AutoViz execution completed


In [35]:
import os
import matplotlib.pyplot as plt

os.makedirs(autoviz_dir, exist_ok=True)

fig_nums = plt.get_fignums()

print(f"Total figures found: {len(fig_nums)}")

for i, fig_num in enumerate(fig_nums):
    fig = plt.figure(fig_num)

    # FORCE single chart isolation
    fig.canvas.draw()

    save_path = os.path.join(autoviz_dir, f"autoviz_chart_{i+1}.png")

    fig.savefig(save_path, bbox_inches="tight", dpi=300)

print(f"✓ Saved {len(fig_nums)} individual charts to: {autoviz_dir}")

Total figures found: 14
✓ Saved 14 individual charts to: reports/winequality-red-edited/autoviz_plots


## 12. Launch D-Tale Interactive Dashboard

In [30]:

print('\n' + '='*80)
print('D-TALE INTERACTIVE DASHBOARD')
print('='*80)

try:
    import dtale
    import warnings
    warnings.filterwarnings('ignore')
    print('\nLaunching D-Tale dashboard...')
    
    d = dtale.show(df, open_browser=False)
    
    print(f'✓ D-Tale is running!')
    print("D-Tale URL:", d._main_url)
    print('\nD-Tale Features:')
    print('  - Interactive data exploration')
    print('  - Column statistics')
    print('  - Filtering & sorting')
    print('  - Visualization tools')
    print('\nNote: Keep this notebook running to access D-Tale.')
    print('\nTo stop D-Tale, uncomment and run the cleanup cell below.')
    
    dtale_instance = d

except ImportError:
    print('⚠ D-Tale not installed.')
    print('  Install: pip install dtale')
except Exception as e:
    print(f'⚠ Error launching D-Tale: {str(e)[:200]}')
    print('  Try updating: pip install --upgrade dtale')



D-TALE INTERACTIVE DASHBOARD

Launching D-Tale dashboard...
✓ D-Tale is running!
D-Tale URL: http://phoenix:40000/dtale/main/2

D-Tale Features:
  - Interactive data exploration
  - Column statistics
  - Filtering & sorting
  - Visualization tools

Note: Keep this notebook running to access D-Tale.

To stop D-Tale, uncomment and run the cleanup cell below.


## 13. Execution Summary

In [31]:
end_time = datetime.now()
execution_time = (end_time - start_time).total_seconds()

print('\n' + '='*80)
print('EXECUTION COMPLETE')
print('='*80)

print(f'\n✓ EDA Pipeline finished successfully!')
print(f'  Execution Time: {execution_time:.2f} seconds')
print(f'  Output Directory: {dataset_output_dir}')

# List generated files
print(f'\nGenerated Reports:')
for item in os.listdir(dataset_output_dir):
    item_path = os.path.join(dataset_output_dir, item)
    if os.path.isfile(item_path):
        size_mb = os.path.getsize(item_path) / 1024**2
        print(f'  ✓ {item} ({size_mb:.2f} MB)')
    elif os.path.isdir(item_path):
        file_count = len([f for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f))])
        print(f'  ✓ {item}/ ({file_count} files)')

print(f'\nOpen these reports in a web browser to explore the data!')


EXECUTION COMPLETE

✓ EDA Pipeline finished successfully!
  Execution Time: 51.56 seconds
  Output Directory: reports/winequality-red-edited

Generated Reports:
  ✓ autoviz_plots/ (7 files)
  ✓ data_profile_report.html (0.01 MB)
  ✓ sweetviz_report.html (1.29 MB)

Open these reports in a web browser to explore the data!


## 14. Optional Cleanup

In [32]:
# Uncomment to stop D-Tale
# if 'dtale_instance' in locals():
#     dtale_instance.kill()
#     print('D-Tale dashboard stopped.')